In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv('StudentsPerformance.csv')
df.head()

In [4]:
print("Records:", df.shape[0])   # shape[0] = number of rows
print("Features:", df.shape[1])  # shape[1] = number of columns
df.dtypes

In [ ]:
print("--- Missing values ---")
print(df.isna().sum())                     # blanks per column
 
print("--- Duplicate rows ---")
print(df.duplicated().sum())               # exact copies of a row

print("--- GENDER ---")
print(df["gender"].value_counts()) #rows per gender spelling
print(df["gender"].unique())              # distinct gender spellings

print("--- RACE AND ETHNICITY ---")
print(df["race/ethnicity"].value_counts()) # rows per race/ethnicity
print(df["race/ethnicity"].unique())         # distinct race/ethnicity values

print("--- PARENTAL LEVEL OF EDUCATION ---")
print(df["parental level of education"].value_counts())  # rows per parental education level
print(df["parental level of education"].unique())  # distinct parental education levels

print("--- LUNCH ---")
print(df["lunch"].value_counts())  # rows per lunch type
print(df["lunch"].unique())  # distinct lunch types

print("--- TEST PREPARATION COURSE ---") 
print(df["test preparation course"].value_counts())  # rows per test preparation course
print(df["test preparation course"].unique())  # distinct test preparation courses

print("--- SUMMARY STATISTICS: GRADES ---")
continuous_cols = ["math score", "reading score", "writing score"]
print(df[continuous_cols].describe())      # stats, real numbers only

print("--- SUMMARY STATISTICS: CATEGORIES ---")
print(df.describe(include="object"))       # stats for text columns


 

# step five

In [35]:
# assumptions and boundaries checking. Identify at least two assumptions you're making about this data, and where possible, check them with code. 
# For example: are all three score columns realistically bounded between 0 and 100? Are the categories in each text column spelled consistently, with no near-duplicates?

# assumption 1: All three score columns (math score, reading score, writing score) are bounded between 0 and 100.
# assumption 2: The categories in each text column are spelled consistently, with no near-duplicates. 

# Check assumption 1:
for col in continuous_cols: # previsouly defined list of continuous columns. 
    if df[col].min() < 0 or df[col].max() > 100:
        print(f"Assumption 1 failed for {col}: values are not bounded between 0 and 100.")
    else:
        print(f"Assumption 1 passed for {col}: values are bounded between 0 and 100.")

# Check assumption 2:
df.select_dtypes(include=["object", "string"]).columns  # get all text columns
# in the previsouly step we have already printed the unique values for each text column therefore assumption 2 is already checked visually, there are no near duplicates. 

In [ ]:
from sklearn.preprocessing import StandardScaler  # rescales features to comparable ranges
from sklearn.cluster import KMeans                 # groups similar rows together
 
features = ["math score", "reading score", "writing score"]  # features to use for clustering
clean = df.dropna(subset=features)                 # drop rows missing any of these
#this commands ensures that only clean data with no NaN values is used. this avoids skewing the data. 
 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clean[features])   # rescale so no feature dominates
#cleans features, so that big or small numbers dont dominate the clustering. this is normalisation of the data. 
 
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10) #n_init = 10 means the algorithm will run 10 times with 
# different initializations and return the best output.
clean["cluster"] = kmeans.fit_predict(X_scaled)     # assign each row to a cluster
 
print(pd.crosstab(clean["cluster"], clean["test preparation course"])) 
# after group clusters of the students score, crosstab with test prep course will aim to show us any relationship between students who prepared or not. 

# group by cluster and calculate mean scores for each cluster
cluster_means = clean.groupby("cluster")[features].mean()
print("Cluster Means:")
print(cluster_means)
# this tells the performance metric for  each cluster. 
# we can now say that cluster 3 is the best performing cluster, cluster 0 is the second best and so on.
# cluster 3 is the best performing because completed > none. Cluster 2 is the worst performing because completed << none. 
# with this observation we can say that test preparation course has a positive impact on student performance.
# we can also say that the closer completed is to none, the better the performance of that cluster. 

In [ ]:
# Your task: Split your data into training and test sets, train a classifier using the three score columns as features, predict on the test set, 
# and calculate accuracy. Then compare your accuracy against the simplest possible baseline — always guessing the most common category — to judge whether 
# your model is actually learning anything.
from sklearn.model_selection import train_test_split  # splits data into train/test sets
from sklearn.tree import DecisionTreeClassifier         # the model we'll train
from sklearn.metrics import accuracy_score               # scores how many predictions were correct
 
X = clean[features]        # inputs (the scores, previously defined)
y = clean["test preparation course"]       # the label we're trying to predict
 
X_train, X_test, y_train, y_test = train_test_split ( # this splits the entries in the csv. so out of 1000 students, 100 will be tested and 900 will train the model. 
    X, y, test_size=0.1, random_state=42, stratify=y
)
# 70% train, 30% test, keep test prep proportions balanced
 
model = DecisionTreeClassifier(max_depth=4, random_state=42) #decision tree model is used to predict the grades based on test prep
# max_depth=4 means the tree can have a maximum of 4 levels, which helps prevent overfitting.
model.fit(X_train, y_train)   # learn patterns from the training data
 
predictions = model.predict(X_test)   # guess students on data it hasn't seen, this is the 300 students sitting in reserve. 
print("Model Accuracy:", round(accuracy_score(y_test, predictions) * 100, 2), "%")

baseline_accuracy = y_test.value_counts(normalize=True).max()  #y_test.value_counts(normalize=True) gives the proportion of each category in the test set (completed or none). 
# .max() gives the proportion of the most common category. so if non is the most common, the baseline picks that. 
print(
    "Baseline accuracy:",
    round(baseline_accuracy * 100, 2),
    "%"
) # this is the accuracy of the model if it always guessed the most common category. my model needs to be better than this baseline to be useful. 
# it is 6% higher than baseline, so it is learning something.

